# 03 — Model Training

Interactive hyperparameter widgets, inline Trainer invocation, and live loss curve display.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import config
from data_loader import load_ptbxl, load_mitbih, DatasetSplitter, ECGDataset
from preprocessor import Preprocessor
from model import build_model
from train import Trainer

%matplotlib inline
config.set_global_seeds(42)

## 1. Hyperparameter widgets

In [ ]:
w_dataset = widgets.Dropdown(
    options=[('PTB-XL', 'ptbxl'), ('MIT-BIH', 'mitbih')],
    value='ptbxl', description='Dataset:'
)
w_model = widgets.Dropdown(
    options=[('1D CNN', 'cnn1d'), ('1D ResNet', 'resnet1d')],
    value='cnn1d', description='Model:'
)
w_epochs = widgets.IntSlider(value=5, min=1, max=100, step=1, description='Epochs:')
w_batch = widgets.IntSlider(value=32, min=8, max=256, step=8, description='Batch size:')
w_lr = widgets.FloatLogSlider(value=1e-3, base=10, min=-5, max=-1, step=0.5, description='LR:')
w_subset = widgets.IntSlider(value=200, min=50, max=2000, step=50, description='Subset N:')
w_focal = widgets.Checkbox(value=False, description='Focal loss')
w_mixed = widgets.Checkbox(value=False, description='Mixed precision')

display(widgets.VBox([
    widgets.HBox([w_dataset, w_model]),
    widgets.HBox([w_epochs, w_batch]),
    widgets.HBox([w_lr, w_subset]),
    widgets.HBox([w_focal, w_mixed]),
]))

## 2. Load data and run training

In [ ]:
run_btn = widgets.Button(description='Start Training', button_style='success', icon='play')
out = widgets.Output()

def on_run(b):
    with out:
        clear_output(wait=True)
        dataset_name = w_dataset.value
        model_name   = w_model.value
        epochs       = w_epochs.value
        batch_size   = w_batch.value
        lr           = w_lr.value
        subset_n     = w_subset.value
        focal        = w_focal.value
        mixed        = w_mixed.value

        print(f'Loading {dataset_name}...')
        try:
            if dataset_name == 'ptbxl':
                ds = load_ptbxl(config.PATHS.ptbxl)
                task = 'multilabel'
            else:
                ds = load_mitbih(config.PATHS.mitbih)
                task = 'multiclass'
        except FileNotFoundError as e:
            print(f'ERROR: {e}'); return

        # Subset
        rng = np.random.default_rng(42)
        idx = rng.choice(len(ds), min(subset_n, len(ds)), replace=False)
        ds_sub = ECGDataset(
            X=ds.X[idx], y=ds.y[idx], labels=ds.labels,
            fs=ds.fs, n_leads=ds.n_leads,
            meta=ds.meta.iloc[idx].reset_index(drop=True) if ds.meta is not None else None
        )

        splitter = DatasetSplitter(seed=42)
        train_ds, val_ds, _ = splitter.split(ds_sub)

        prep = Preprocessor(fs=train_ds.fs, target_fs=config.SIGNAL.target_fs)
        X_train = prep.fit_transform(train_ds.X)
        X_val   = prep.transform(val_ds.X)

        n_leads     = X_train.shape[1]
        n_timesteps = X_train.shape[2]
        num_classes = len(ds.labels)

        print(f'Building {model_name}: n_leads={n_leads}, n_t={n_timesteps}, n_classes={num_classes}, task={task}')
        model = build_model(model_name, n_leads, n_timesteps, num_classes, task=task)

        trainer = Trainer(
            model=model, dataset_name=dataset_name,
            batch_size=batch_size, epochs=epochs,
            learning_rate=lr, use_focal_loss=focal,
            mixed_precision=mixed,
        )

        history = trainer.train(X_train, train_ds.y, X_val, val_ds.y)

        # Live loss plot
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(history.history['loss'], label='train loss')
        axes[0].plot(history.history.get('val_loss', []), label='val loss')
        axes[0].set_title('Loss'); axes[0].legend()

        if 'auc' in history.history:
            axes[1].plot(history.history['auc'], label='train AUC')
            axes[1].plot(history.history.get('val_auc', []), label='val AUC')
            axes[1].set_title('AUC'); axes[1].legend()

        plt.tight_layout()
        plt.show()
        print('Training complete!')

run_btn.on_click(on_run)
display(run_btn, out)